# 01 Exploration

Load all local CRMLS sold CSV files from `data/raw/` and run initial exploration for the California close price prediction project.

In [ ]:
# Import Libraries
import math
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import numpy as np

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 100)

sns.set_theme(style="whitegrid")

# 1. Dataset Overview

### Load Data

In [ ]:
# Load every local CRMLS sold CSV file.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"

csv_files = sorted(RAW_DATA_DIR.glob("CRMLSSold*.csv"))

print(f"Found {len(csv_files)} CRMLSSold CSV files")

for path in csv_files:
    print(path.name)

In [ ]:
# Load a CRMLS CSV and add source_file so each row can be traced back to its original file.
def load_crmls_file(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, low_memory=False)
    df["source_file"] = path.name
    return df


# Store each monthly file separately first, then concatenate into one analysis DataFrame.
dataframes = []
file_summary = []

for path in csv_files:
    df = load_crmls_file(path)
    dataframes.append(df)

    # Keep a compact file-level summary to check row counts and schema consistency across files.
    file_summary.append(
        {
            "source_file": path.name,
            "rows": len(df),
            "columns": df.shape[1],
        }
    )

# Combine all monthly files into one raw dataset.
raw_df = pd.concat(dataframes, ignore_index=True)
file_summary_df = pd.DataFrame(file_summary)

print(f"Combined shape: {raw_df.shape}")
file_summary_df

In [ ]:
# Inspect column names, inferred dtypes, non-null counts, and approximate memory usage.
raw_df.info(memory_usage="deep")

### Filter to Project Scope

The task prompt says to model only records where `PropertyType == "Residential"` and `PropertySubType == "SingleFamilyResidence"`.

In [ ]:
# Check the main filtering fields.
for col in ["PropertyType", "PropertySubType", "MlsStatus"]:
    if col in raw_df.columns:
        display(raw_df[col].value_counts(dropna=False).head(20).to_frame("count"))

In [ ]:
# Filter the dataset.
df = raw_df[
    (raw_df["PropertyType"] == "Residential") &
    (raw_df["PropertySubType"].astype(str).str.replace(" ", "", regex=False) == "SingleFamilyResidence")
].copy()


print(f"Original rows: {raw_df.shape[0]:,}")
print(f"Filtered rows: {df.shape[0]:,}")
print(f"Remaining ratio: {df.shape[0] / raw_df.shape[0]:.2%}")

# 2. Data Quality Checks

### Duplicate ListingKey

In [ ]:
# Count duplicate ListingKey rows.
# ListingKey should generally identify one listing, so duplicates need follow-up before modeling.
df["ListingKey"].duplicated().sum()

In [ ]:
# Summarize duplicate ListingKeys
# n_rows: how many records share the same key
# n_files: whether duplicates span files
duplicated_listing_keys = (
    df.dropna(subset=["ListingKey"])
    .groupby("ListingKey")
    .agg(
        n_rows=("ListingKey", "size"),
        n_files=("source_file", "nunique"),
        files=("source_file", lambda x: ", ".join(sorted(x.unique()))),
    )
    .query("n_rows > 1")
    .sort_values(["n_rows", "n_files"], ascending=False)
)

print(f"Number of ListingKeys with duplicate rows: {len(duplicated_listing_keys):,}")
print(f"Number of extra rows caused by duplicated ListingKeys: {(duplicated_listing_keys['n_rows'] - 1).sum():,}")

# Classify duplicates (repeated within a single file or repeated across multiple files).
duplicate_type_summary = (
    duplicated_listing_keys.assign(
        duplicate_type=lambda x: x["n_files"].gt(1).map(
            {True: "across_files", False: "within_same_file"}
        )
    )["duplicate_type"]
    .value_counts()
    .to_frame("n_listing_keys")
)

# Display a summary and examples.
display(duplicate_type_summary)
display(duplicated_listing_keys.head(20))

In [ ]:
# Pull the full duplicate rows so we can compare whether important fields differ.
duplicated_key_values = duplicated_listing_keys.index
duplicated_rows = df[df["ListingKey"].isin(duplicated_key_values)].copy()

# Keep only fields that help diagnose whether duplicates are exact repeats or updated records.
columns_to_compare = [
    "source_file",
    "CloseDate",
    "ClosePrice",
    "MlsStatus",
    "ListPrice",
    "PropertyType",
    "PropertySubType",
    "UnparsedAddress",
    "City",
    "PostalCode",
]
columns_to_compare = [col for col in columns_to_compare if col in duplicated_rows.columns]

# Count how many distinct values each duplicated ListingKey has in key fields.
# Values greater than 1 mean that the duplicate rows are not identical in that field.
variation_summary = (
    duplicated_rows.groupby("ListingKey")
    .agg(
        n_rows=("ListingKey", "size"),
        n_files=("source_file", "nunique"),
        n_close_dates=("CloseDate", "nunique"),
        n_close_prices=("ClosePrice", "nunique"),
        n_statuses=("MlsStatus", "nunique"),
        n_list_prices=("ListPrice", "nunique"),
    )
    .sort_values(["n_close_prices", "n_close_dates", "n_rows"], ascending=False)
)

display(variation_summary.head(20))

# Show one concrete example so the duplicate pattern can be inspected manually.
if len(variation_summary) > 0:
    sample_key = variation_summary.index[0]
    print(f"Sample duplicated ListingKey: {sample_key}")
    display(
        duplicated_rows.loc[
            duplicated_rows["ListingKey"] == sample_key,
            ["ListingKey"] + columns_to_compare,
        ].sort_values(["source_file", "CloseDate"])
    )

### Coordinates

In [ ]:
# Quick coordinate scatter plot to confirm the records are located in California.
plt.figure(figsize=(7, 6))
plt.scatter(df["Longitude"], df["Latitude"], s=2, alpha=0.3)
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.title("Property Locations")
plt.show()

In [ ]:
display(df[["Latitude", "Longitude"]].describe())

# Flag coordinates outside the expected California range.
# These records may need to be removed or have coordinates set to missing before modeling.
ca_lat_min, ca_lat_max = 32.0, 42.5
ca_lon_min, ca_lon_max = -125.0, -114.0

outside_ca = df[
    (df["Latitude"] < ca_lat_min) |
    (df["Latitude"] > ca_lat_max) |
    (df["Longitude"] < ca_lon_min) |
    (df["Longitude"] > ca_lon_max)
]

print(f"Number of properties outside California: {outside_ca.shape[0]}")

# 3. Check Target variable: ClosePrice

In [ ]:
# Convert the target variable to numeric.
df["ClosePrice"] = pd.to_numeric(df["ClosePrice"], errors="coerce")

In [ ]:
# Check whether the target has missing, invalid, or non-positive values.
print(f"Number of missing values in ClosePrice: {df['ClosePrice'].isna().sum()}")
print(f"Number of non-positive values in ClosePrice: {(df['ClosePrice'] <= 0).sum()}")
print(f"Description of ClosePrice:\n{df['ClosePrice'].describe()}")

In [ ]:
# Use percentile clipping only for plotting the target distribution.
# This keeps extreme luxury sales from compressing the main body of the histogram.
lower = df["ClosePrice"].quantile(0.01)
upper = df["ClosePrice"].quantile(0.99)

plt.figure(figsize=(8, 5))
df["ClosePrice"].dropna().clip(lower, upper).hist(bins=50)
plt.xlabel("ClosePrice")
plt.ylabel("Count")
plt.title("Distribution of ClosePrice (Clipped at 1st and 99th Percentiles)")
plt.show()

In [ ]:
# Plot log-transformed ClosePrice to see whether the target becomes more symmetric.
# log1p is commonly used for right-skewed price targets.
plt.figure(figsize=(8, 5))
np.log1p(df["ClosePrice"].dropna()).hist(bins=50)
plt.xlabel("log(1 + ClosePrice)")
plt.ylabel("Count")
plt.title("Distribution of log(ClosePrice)")
plt.show()

## 4. Missing Values and Candidate Feature Groups

In [ ]:
# Calculate missing value rate for every column after filtering to the project scope.
# Keep both a detailed table and a compact grouped summary for presentation.
missing = (
    df.isna()
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)
missing.columns = ["column", "missing_rate"]

missing["missing_group"] = pd.cut(
    missing["missing_rate"],
    bins=[-0.01, 0, 0.05, 0.25, 0.50, 0.80, 1.00],
    labels=["0%", "0-5%", "5-25%", "25-50%", "50-80%", "80-100%"],
)

missing_group_summary = (
    missing["missing_group"]
    .value_counts(sort=False)
    .rename_axis("missing_rate_group")
    .reset_index(name="n_columns")
)

# This table is good for slides because it summarizes missingness without listing every column.
display(missing_group_summary)

# Keep the detailed top missing columns for preprocessing decisions.
display(missing.head(30))

In [ ]:
# Draft feature lists for the next preprocessing/modeling notebook.
drop_cols = [
    # Target
    "ClosePrice",

    # leakage
    "ListPrice",
    "OriginalListPrice",
    "CloseDate",

    # IDs
    "ListingKey",
    "ListingKeyNumeric",
    "ListingId",

    # 100% missing
    "CoveredSpaces",
    "MiddleOrJuniorSchoolDistrict",
    "TaxYear",
    "ElementarySchoolDistrict",
    "BusinessType",
    "TaxAnnualAmount",
    "FireplacesTotal",
    "AboveGradeFinishedArea",

    # Very high missing (> 80%)
    "WaterfrontYN",
    "BelowGradeFinishedArea",
    "BasementYN",
    "BuilderName",
    "LotSizeDimensions",
    "BuildingAreaTotal",
    "CoBuyerAgentFirstName",
    "MiddleOrJuniorSchool",
    "ElementarySchool",
    "HighSchool",

    # Agent / office fields
    "ListAgentEmail",
    "ListAgentFirstName",
    "ListAgentLastName",
    "ListAgentFullName",
    "CoListAgentFirstName",
    "CoListAgentLastName",
    "BuyerAgentFirstName",
    "BuyerAgentLastName",
    "BuyerAgentMlsId",
    "ListOfficeName",
    "BuyerOfficeName",
    "CoListOfficeName",
    "BuyerAgentAOR",
    "ListAgentAOR",
    "BuyerOfficeAOR",

    # Transaction timing / potential leakage for first model
    "PurchaseContractDate",
    "ContractStatusChangeDate",
    "DaysOnMarket",

    # Too sparse or not needed initially
    "SubdivisionName",
    "AssociationFeeFrequency",
    "MainLevelBedrooms",    # 39% missing + 'BedroomsTotal' already provides similar information.
    "StateOrProvince"
]

# Candidate numeric predictors for initial EDA and baseline modeling.
numeric_cols = [
    "LivingArea",
    "BedroomsTotal",
    "BathroomsTotalInteger",
    "LotSizeSquareFeet",
    "YearBuilt",
    "Latitude",
    "Longitude",
    "GarageSpaces",
    "ParkingTotal",
    "Stories",
    "AssociationFee"
]

# Boolean-like predictors that may need conversion from strings/YN values before modeling.
boolean_cols = [
    "ViewYN",
    "PoolPrivateYN",
    "AttachedGarageYN",
    "FireplaceYN",
    "NewConstructionYN"
]

# Categorical predictors that may need encoding or rare-category grouping.
categorical_cols = [
    "City",
    "CountyOrParish",
    "PostalCode",
    "MLSAreaMajor",
    "Levels",
    "Flooring",
    "HighSchoolDistrict"
]

### Numeric Feature Summary

In [ ]:
# Convert candidate numeric features to numeric dtype.
# Invalid strings become NaN so summaries and plots do not fail.
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

available_numeric_cols = [col for col in numeric_cols if col in df.columns]

# Compact numeric summary for preprocessing decisions.
# Missing rate indicates imputation needs; skew helps decide whether log transforms may be useful.
numeric_summary = df[available_numeric_cols].agg([
    "count",
    "mean",
    "median",
    "std",
    "min",
    "max",
]).T

numeric_summary["missing_rate"] = df[available_numeric_cols].isna().mean()
numeric_summary["skew"] = df[available_numeric_cols].skew(numeric_only=True)

# Reorder columns so the most useful EDA fields appear first.
numeric_summary = numeric_summary[
    ["count", "missing_rate", "mean", "median", "std", "min", "max", "skew"]
]

display(numeric_summary)

In [ ]:
# Plot only the core numeric features for a concise EDA section.
# These are the most interpretable predictors for the first modeling pass.
core_numeric_plot_cols = [
    "LivingArea",
    "BedroomsTotal",
    "BathroomsTotalInteger",
    "LotSizeSquareFeet",
    "YearBuilt",
]
core_numeric_plot_cols = [col for col in core_numeric_plot_cols if col in df.columns]

fig, axes = plt.subplots(len(core_numeric_plot_cols), 2, figsize=(13, 4 * len(core_numeric_plot_cols)))

for i, col in enumerate(core_numeric_plot_cols):
    values = df[col].dropna()

    # Clip the original-scale histogram only for readability.
    # The raw values remain unchanged in df.
    lower = values.quantile(0.01)
    upper = values.quantile(0.99)
    clipped_values = values.clip(lower, upper)

    sns.histplot(clipped_values, bins=50, ax=axes[i, 0])
    axes[i, 0].set_title(f"{col} Distribution")
    axes[i, 0].set_xlabel(f"{col} (clipped at 1st/99th pct)")
    axes[i, 0].set_ylabel("Count")

    # Log histogram is useful for right-skewed non-negative variables.
    non_negative_values = values[values >= 0]
    sns.histplot(np.log1p(non_negative_values), bins=50, ax=axes[i, 1])
    axes[i, 1].set_title(f"log(1 + {col}) Distribution")
    axes[i, 1].set_xlabel(f"log(1 + {col})")
    axes[i, 1].set_ylabel("Count")

plt.tight_layout()
plt.show()

### Categorical Feature Summary

In [ ]:
# Summarize categorical columns by cardinality, missingness, and most common category.
# This informs encoding choices for preprocessing.
cat_summary = []

for col in categorical_cols:
    if col in df.columns:
        top_values = df[col].value_counts(dropna=False).head(5)
        cat_summary.append({
            "column": col,
            "n_unique": df[col].nunique(dropna=True),
            "missing_rate": df[col].isna().mean(),
            "top_value": top_values.index[0],
            "top_value_count": top_values.iloc[0],
            "top_5_values": ", ".join(map(str, top_values.index.tolist())),
        })

cat_summary_df = pd.DataFrame(cat_summary).sort_values("n_unique", ascending=False)
display(cat_summary_df)

# Show compact top-category tables for the most useful location/category fields.
# This is easier to interpret than plotting every categorical variable.
for col in ["CountyOrParish", "City", "PostalCode", "MLSAreaMajor"]:
    if col in df.columns:
        print(f"\nTop 10 values for {col}")
        display(df[col].value_counts(dropna=False).head(10).to_frame("count"))

### Location Summary

In [ ]:
# County-level summary combines sample size and price level.
# This confirms that geography is likely a major predictor of ClosePrice.
county_summary = (
    df.dropna(subset=["CountyOrParish", "ClosePrice"])
    .groupby("CountyOrParish")
    .agg(
        n_sales=("ClosePrice", "size"),
        median_close_price=("ClosePrice", "median"),
        mean_close_price=("ClosePrice", "mean"),
    )
    .sort_values("n_sales", ascending=False)
)

print("Counties with the most sales")
display(county_summary.head(15))

print("Counties with the highest median ClosePrice, minimum 50 sales")
display(
    county_summary
    .query("n_sales >= 50")
    .sort_values("median_close_price", ascending=False)
    .head(15)
)

In [ ]:
# ZIP code is more granular than county but has high cardinality.
# Use a minimum sales threshold so the ranking is not dominated by tiny samples.
zip_summary = (
    df.dropna(subset=["PostalCode", "ClosePrice"])
    .groupby("PostalCode")
    .agg(
        n_sales=("ClosePrice", "size"),
        median_close_price=("ClosePrice", "median"),
    )
    .sort_values("n_sales", ascending=False)
)

print("ZIP codes with the most sales")
display(zip_summary.head(15))

print("ZIP codes with the highest median ClosePrice, minimum 30 sales")
display(
    zip_summary
    .query("n_sales >= 30")
    .sort_values("median_close_price", ascending=False)
    .head(15)
)

# 5. Key features vs. Target variable

In [ ]:
# Define model-relevant numeric features to compare directly with the target.
# This section focuses on interpretable property characteristics and location fields.
key_features = [
    "LivingArea",
    "BedroomsTotal",
    "BathroomsTotalInteger",
    "LotSizeSquareFeet",
    "YearBuilt",
    "Latitude",
    "Longitude",
    "GarageSpaces",
    "ParkingTotal",
    "Stories",
    "AssociationFee",
]

key_features = [col for col in key_features if col in df.columns]

# Build a numeric-only analysis frame for correlations and plots.
feature_target_df = df[["ClosePrice"] + key_features].copy()

for col in feature_target_df.columns:
    feature_target_df[col] = pd.to_numeric(feature_target_df[col], errors="coerce")

# Keep only positive target values for target-feature relationship checks.
feature_target_df = feature_target_df[feature_target_df["ClosePrice"].gt(0)].copy()

# Compare Pearson and Spearman correlations.
# Spearman is often more useful in EDA because it captures monotonic relationships and is less sensitive to scale.
corr_summary = pd.DataFrame(
    {
        "pearson_corr": feature_target_df.corr(numeric_only=True)["ClosePrice"],
        "spearman_corr": feature_target_df.corr(method="spearman", numeric_only=True)["ClosePrice"],
        "missing_rate": feature_target_df.isna().mean(),
    }
).drop(index="ClosePrice")

corr_summary["abs_spearman_corr"] = corr_summary["spearman_corr"].abs()
corr_summary = corr_summary.sort_values("abs_spearman_corr", ascending=False)

display(corr_summary.drop(columns="abs_spearman_corr"))

# Use clipping and sampling for readable, fast plots. This affects plots only, not the raw data.
plot_df = feature_target_df.sample(
    n=min(15000, len(feature_target_df)),
    random_state=42,
).copy()

price_upper = feature_target_df["ClosePrice"].quantile(0.99)
plot_df["ClosePrice_clipped"] = plot_df["ClosePrice"].clip(upper=price_upper)

# Continuous features are shown with scatter plots.
continuous_features = [
    "LivingArea",
    "LotSizeSquareFeet",
    "YearBuilt",
    "Latitude",
    "Longitude",
    "AssociationFee",
]
continuous_features = [col for col in continuous_features if col in key_features]

n_cols = 3
n_rows = math.ceil(len(continuous_features) / n_cols)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4 * n_rows))
axes = np.array(axes).reshape(-1)

for i, col in enumerate(continuous_features):
    # Clip x values only in the plot so extreme values do not hide the main relationship.
    x_upper = feature_target_df[col].quantile(0.99)
    x_values = plot_df[col].clip(upper=x_upper)

    sns.scatterplot(
        x=x_values,
        y=plot_df["ClosePrice_clipped"],
        alpha=0.2,
        s=12,
        ax=axes[i],
    )
    axes[i].set_title(f"ClosePrice vs {col}")
    axes[i].set_xlabel(f"{col} (clipped at 99th pct)")
    axes[i].set_ylabel("ClosePrice (clipped at 99th pct)")

for j in range(len(continuous_features), len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()

# Discrete/count-like features are shown with boxplots.
discrete_features = [
    "BedroomsTotal",
    "BathroomsTotalInteger",
    "GarageSpaces",
    "ParkingTotal",
    "Stories",
]
discrete_features = [col for col in discrete_features if col in key_features]

n_cols = 3
n_rows = math.ceil(len(discrete_features) / n_cols)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4 * n_rows))
axes = np.array(axes).reshape(-1)

for i, col in enumerate(discrete_features):
    temp = plot_df[[col, "ClosePrice_clipped"]].dropna().copy()
    cap = feature_target_df[col].quantile(0.99)
    temp[col] = temp[col].clip(upper=cap)

    sns.boxplot(data=temp, x=col, y="ClosePrice_clipped", ax=axes[i], showfliers=False)
    axes[i].set_title(f"ClosePrice by {col}")
    axes[i].set_xlabel(f"{col} (capped at 99th pct)")
    axes[i].set_ylabel("ClosePrice (clipped at 99th pct)")
    axes[i].tick_params(axis="x", rotation=45)

for j in range(len(discrete_features), len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()

# Bin skewed continuous features into deciles and compare median target by bin.
# This makes nonlinear relationships easier to read than a dense scatter plot.
binned_features = ["LivingArea", "LotSizeSquareFeet", "YearBuilt"]
binned_features = [col for col in binned_features if col in key_features]

binned_summaries = []

for col in binned_features:
    temp = feature_target_df[[col, "ClosePrice"]].dropna().copy()
    temp = temp[temp[col].between(temp[col].quantile(0.01), temp[col].quantile(0.99))]
    temp["bin"] = pd.qcut(temp[col], q=10, duplicates="drop")

    summary = (
        temp.groupby("bin", observed=True)
        .agg(
            feature=(col, "median"),
            n_sales=("ClosePrice", "size"),
            median_close_price=("ClosePrice", "median"),
        )
        .reset_index(drop=True)
    )
    summary.insert(0, "feature_name", col)
    binned_summaries.append(summary)

binned_summary_df = pd.concat(binned_summaries, ignore_index=True)
display(binned_summary_df)

# 6. Additional EDA Checks

These checks focus on model-relevant relationships and data quality questions that should be resolved before preprocessing/modeling.

### Price per Square Foot

`ClosePrice / LivingArea` is useful for comparing properties with different sizes. It should only be computed when both `ClosePrice` and `LivingArea` are positive.

EDA only, not a model feature

In [ ]:
# Create a normalized price metric so properties of different sizes can be compared.
# Only compute this when both ClosePrice and LivingArea are positive; otherwise set it to NaN.
df["PricePerSqFt"] = np.where(
    (df["ClosePrice"] > 0) & (df["LivingArea"] > 0),
    df["ClosePrice"] / df["LivingArea"],
    np.nan,
)

# Summarize the full distribution, including tail percentiles.
# The 1st and 99th percentiles help identify suspiciously low/high price-per-sq-ft values.
price_per_sqft_summary = df["PricePerSqFt"].describe(
    percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]
)

display(price_per_sqft_summary.to_frame("PricePerSqFt"))

# Clip only for visualization so extreme values do not make the histogram unreadable.
# This does not change df["PricePerSqFt"] or remove any rows from the dataset.
psf_lower = df["PricePerSqFt"].quantile(0.01)
psf_upper = df["PricePerSqFt"].quantile(0.99)

plt.figure(figsize=(8, 5))
sns.histplot(df["PricePerSqFt"].clip(psf_lower, psf_upper), bins=60)
plt.xlabel("Price per Square Foot (clipped at 1st and 99th percentiles)")
plt.ylabel("Count")
plt.title("Distribution of Price per Square Foot")
plt.tight_layout()
plt.show()

In [ ]:
# Compare price per square foot by county.
# Require at least 50 sales so the ranking is not driven by counties with only a few records.
psf_by_county = (
    df.dropna(subset=["CountyOrParish", "PricePerSqFt"])
    .groupby("CountyOrParish")
    .agg(
        n_sales=("PricePerSqFt", "size"),
        median_price_per_sqft=("PricePerSqFt", "median"),
        median_close_price=("ClosePrice", "median"),
    )
    .query("n_sales >= 50")
    .sort_values("median_price_per_sqft", ascending=False)
)

# Display the most expensive counties on a size-adjusted basis.
display(psf_by_county.head(20))

# Plot the same ranking so regional differences are easier to scan visually.
plt.figure(figsize=(10, 6))
sns.barplot(
    data=psf_by_county.head(20).reset_index(),
    y="CountyOrParish",
    x="median_price_per_sqft",
    color="steelblue",
)
plt.xlabel("Median Price per Square Foot")
plt.ylabel("County")
plt.title("Top Counties by Median Price per Square Foot")
plt.tight_layout()
plt.show()

### County and City Median Price

Use a minimum sales threshold so small cities/counties do not dominate the ranking because of a few unusual transactions.

In [ ]:
# Summarize median sale price by county and city.
# n_sales is kept in the table so we can judge whether each median is based on enough data.
county_price_summary = (
    df.dropna(subset=["CountyOrParish", "ClosePrice"])
    .groupby("CountyOrParish")
    .agg(
        n_sales=("ClosePrice", "size"),
        median_close_price=("ClosePrice", "median"),
        median_price_per_sqft=("PricePerSqFt", "median"),
    )
    .query("n_sales >= 50")
    .sort_values("median_close_price", ascending=False)
)

# City has higher cardinality than county, so the minimum sales threshold is especially important here.
city_price_summary = (
    df.dropna(subset=["City", "ClosePrice"])
    .groupby("City")
    .agg(
        n_sales=("ClosePrice", "size"),
        median_close_price=("ClosePrice", "median"),
        median_price_per_sqft=("PricePerSqFt", "median"),
    )
    .query("n_sales >= 50")
    .sort_values("median_close_price", ascending=False)
)

# Show the top rows from both summaries before plotting.
display(county_price_summary.head(20))
display(city_price_summary.head(20))

In [ ]:
# Plot county and city medians side by side.
# These charts help identify high-price regions that may need strong geographic features in the model.
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

sns.barplot(
    data=county_price_summary.head(15).reset_index(),
    y="CountyOrParish",
    x="median_close_price",
    color="steelblue",
    ax=axes[0],
)
axes[0].set_title("Top Counties by Median ClosePrice")
axes[0].set_xlabel("Median ClosePrice")
axes[0].set_ylabel("County")

sns.barplot(
    data=city_price_summary.head(15).reset_index(),
    y="City",
    x="median_close_price",
    color="darkseagreen",
    ax=axes[1],
)
axes[1].set_title("Top Cities by Median ClosePrice")
axes[1].set_xlabel("Median ClosePrice")
axes[1].set_ylabel("City")

plt.tight_layout()
plt.show()

In [ ]:
# Build a map-ready sample with valid coordinates and target values.
# The California bounds remove obvious coordinate errors before plotting.
map_sample = df.dropna(subset=["Latitude", "Longitude", "ClosePrice"]).copy()
map_sample = map_sample[
    map_sample["Latitude"].between(32.0, 42.5)
    & map_sample["Longitude"].between(-125.0, -114.0)
]

# Sample the rows so the scatter plot stays responsive in the notebook.
map_sample = map_sample.sample(n=min(20000, len(map_sample)), random_state=42)

# Clip color values only for visualization; otherwise a few luxury sales dominate the color scale.
map_sample["ClosePrice_clipped"] = map_sample["ClosePrice"].clip(
    upper=df["ClosePrice"].quantile(0.99)
)

plt.figure(figsize=(8, 7))
scatter = plt.scatter(
    map_sample["Longitude"],
    map_sample["Latitude"],
    c=map_sample["ClosePrice_clipped"],
    s=4,
    alpha=0.45,
    cmap="viridis",
)
plt.colorbar(scatter, label="ClosePrice (clipped at 99th percentile)")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.title("California Property Locations Colored by ClosePrice")
plt.tight_layout()
plt.show()

### Time Coverage and Future Test Set


In [ ]:
# Convert important date fields to datetime.
# These fields help us understand time coverage and identify potential leakage columns.
date_cols = [
    "CloseDate",
    "ListingContractDate",
    "PurchaseContractDate",
    "ContractStatusChangeDate",
]

for col in date_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")

# Check date ranges for each available date column.
# CloseDate defines the chronological split; later transaction dates may be leakage for modeling.
available_date_cols = [col for col in date_cols if col in df.columns]
date_range_summary = df[available_date_cols].agg(["min", "max"]).T

display(date_range_summary)

# Create a monthly close period for trend analysis and future train/test splitting.
df["CloseMonth"] = df["CloseDate"].dt.to_period("M").astype("string")

# Summarize monthly sample size and price levels.
# This helps decide whether the latest month is large enough and stable enough to use as a test set.
monthly_summary = (
    df.dropna(subset=["CloseMonth"])
    .groupby("CloseMonth")
    .agg(
        n_sales=("ClosePrice", "size"),
        median_close_price=("ClosePrice", "median"),
        mean_close_price=("ClosePrice", "mean"),
        median_price_per_sqft=("PricePerSqFt", "median"),
    )
    .reset_index()
    .sort_values("CloseMonth")
)

latest_month = monthly_summary["CloseMonth"].iloc[-1]
latest_month_rows = int(monthly_summary.loc[monthly_summary["CloseMonth"] == latest_month, "n_sales"].iloc[0])

print(f"Latest month available: {latest_month}")
print(f"Rows in latest month: {latest_month_rows:,}")

# Show recent months only to keep the train/test split decision focused.
display(monthly_summary.tail(18))

In [ ]:
# Plot recent monthly sales volume and median price together.
# A sharp drop in the latest month could mean the month is incomplete and should not be used as test data.
recent_months = monthly_summary.tail(18)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sns.barplot(data=recent_months, x="CloseMonth", y="n_sales", color="steelblue", ax=axes[0])
axes[0].set_title("Recent Monthly Row Counts")
axes[0].set_xlabel("Close Month")
axes[0].set_ylabel("Number of Sales")
axes[0].tick_params(axis="x", rotation=45)

sns.lineplot(data=recent_months, x="CloseMonth", y="median_close_price", marker="o", ax=axes[1])
axes[1].set_title("Recent Monthly Median ClosePrice")
axes[1].set_xlabel("Close Month")
axes[1].set_ylabel("Median ClosePrice")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

### Extreme Values in Key Numeric Features

These rows help distinguish true luxury or unusual properties from likely data errors before modeling.

In [ ]:
# Select identifying and model-relevant fields for inspecting extreme numeric values.
# These rows are for manual review only; no rows are removed in this EDA notebook.
extreme_check_cols = [
    "ListingKey",
    "source_file",
    "CloseDate",
    "ClosePrice",
    "LivingArea",
    "LotSizeSquareFeet",
    "YearBuilt",
    "BedroomsTotal",
    "BathroomsTotalInteger",
    "City",
    "CountyOrParish",
    "PostalCode",
]
extreme_check_cols = [col for col in extreme_check_cols if col in df.columns]

# Inspect both tails for key numeric fields.
# Low values can reveal placeholder or missing-data encodings; high values can be true luxury/outlier records.
for feature in ["LivingArea", "LotSizeSquareFeet", "YearBuilt"]:
    if feature not in df.columns:
        continue

    print(f"\nLowest {feature} values")
    display(
        df.dropna(subset=[feature])
        .sort_values(feature, ascending=True)
        .loc[:, extreme_check_cols]
        .head(10)
    )

    print(f"Highest {feature} values")
    display(
        df.dropna(subset=[feature])
        .sort_values(feature, ascending=False)
        .loc[:, extreme_check_cols]
        .head(10)
    )

## EDA Summary

- Loaded 6 CRMLS sold CSV files covering `2025-12` to `2026-05`.
- Raw data contains `124,404` rows. After filtering to `Residential / SingleFamilyResidence`, `61,727` rows remain.
- `ClosePrice` has no missing or non-positive values, but it is highly right-skewed:
  - Median: about `$890K`.
  - Mean: about `$1.34M`.
  - 99th percentile: about `$6.5M`.
  - Maximum: about `$796M`, likely a data issue.
- Key data quality issues to handle before modeling:
  - `62` duplicated `ListingKey` values, creating `70` extra rows.
  - `24` records have coordinates outside the expected California range.
  - `LivingArea <= 0` and `LotSizeSquareFeet <= 0` appear in some records.
  - `LotSizeSquareFeet` and `ClosePrice` contain extreme outliers.
- Missingness varies across columns. High-missing columns, IDs, agent/office fields, and leakage-prone transaction fields should be excluded from the baseline model.
- Strong initial predictors include `LivingArea`, `BathroomsTotalInteger`, `BedroomsTotal`, garage/parking variables, and location-related features.
- Geography is a major price driver. High-price areas include Bay Area and coastal markets such as San Mateo, Santa Clara, Orange, Beverly Hills, Malibu, and Newport Beach.
- Latest available month is `2026-05` with `12,024` rows, so it is a reasonable candidate for a chronological test set.
- Modeling plan:
  - Deduplicate by `ListingKey`.
  - Clean obvious numeric errors and outliers.
  - Engineer location and property-age features.
  - Use previous months for training and `2026-05` for testing.
  - Compare models using raw `ClosePrice` and `log1p(ClosePrice)` as target.
